# LangGraph — Sports latest news + persistent storage

- **LangGraph `SqliteSaver`**: checkpoints / thread state in `sports_checkpoints.db`.
- **App SQLite DB** (`sports_news.db`): stores each assistant reply row in `news_items` for retrieval.

Set `OPENAI_API_KEY` and **`SERPER_API_KEY`** (from [serper.dev](https://serper.dev)) in `.env`. Uses **`langchain_community`** `GoogleSerperAPIWrapper` — no extra pip packages. Run the notebook from this folder so DB paths resolve correctly.

In [13]:
from __future__ import annotations

import sqlite3
from datetime import datetime, timezone
from pathlib import Path
from typing import Annotated, Any, Literal

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing_extensions import TypedDict

load_dotenv(override=True)

# DB files next to this notebook (set cwd to this directory when running)
BASE = Path.cwd()
NEWS_DB = BASE / "sports_news.db"
CHECKPOINT_DB = BASE / "sports_checkpoints.db"


# Must be module-level so LangGraph can resolve type hints on routing functions (get_type_hints).
class SportsNewsState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


## 1. News store (SQLite, separate from LangGraph checkpoints)

In [14]:
def _news_connect() -> sqlite3.Connection:
    conn = sqlite3.connect(str(NEWS_DB), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def init_news_db() -> sqlite3.Connection:
    conn = _news_connect()
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS news_items (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            thread_id TEXT NOT NULL,
            sport TEXT,
            query TEXT,
            raw_summary TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
        CREATE INDEX IF NOT EXISTS idx_news_thread ON news_items(thread_id);
        CREATE INDEX IF NOT EXISTS idx_news_created ON news_items(created_at);
        """
    )
    conn.commit()
    return conn


def save_fetch(
    thread_id: str,
    raw_summary: str,
    *,
    sport: str | None = None,
    query: str | None = None,
) -> int:
    conn = init_news_db()
    try:
        cur = conn.execute(
            """
            INSERT INTO news_items (thread_id, sport, query, raw_summary, created_at)
            VALUES (?, ?, ?, ?, ?)
            """,
            (
                thread_id,
                sport,
                query,
                raw_summary,
                datetime.now(timezone.utc).isoformat(),
            ),
        )
        conn.commit()
        return int(cur.lastrowid)
    finally:
        conn.close()


def list_recent(limit: int = 20) -> list[dict]:
    conn = init_news_db()
    try:
        rows = conn.execute(
            """
            SELECT id, thread_id, sport, query, raw_summary, created_at
            FROM news_items
            ORDER BY id DESC
            LIMIT ?
            """,
            (limit,),
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


def list_for_thread(thread_id: str, limit: int = 50) -> list[dict]:
    conn = init_news_db()
    try:
        rows = conn.execute(
            """
            SELECT id, thread_id, sport, query, raw_summary, created_at
            FROM news_items
            WHERE thread_id = ?
            ORDER BY id DESC
            LIMIT ?
            """,
            (thread_id, limit),
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()

## 2. Tools + graph (Serper web search → LLM → persist)

In [15]:
_serper = GoogleSerperAPIWrapper()


@tool
def search_latest_sports_news(
    sport: str = "general",
    extra_terms: str = "",
) -> str:
    """Search the web for the latest sports news. Use sport like 'soccer', 'NBA', 'tennis', or 'general'."""
    sport = sport.strip() or "general"
    q = f"latest {sport} sports news today headlines"
    if extra_terms.strip():
        q = f"{q} {extra_terms.strip()}"
    try:
        return _serper.run(q)
    except Exception as e:
        return f"Search failed: {e!s}"


def make_checkpointer(db_path: Path | None = None) -> SqliteSaver:
    path = db_path or CHECKPOINT_DB
    conn = sqlite3.connect(str(path), check_same_thread=False)
    return SqliteSaver(conn)


def build_graph(checkpointer: Any):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    tools = [search_latest_sports_news]
    llm_with_tools = llm.bind_tools(tools)

    def sports_chat(state: SportsNewsState) -> dict[str, Any]:
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    tool_node = ToolNode(tools)

    def persist_after_turn(state: SportsNewsState, config: RunnableConfig) -> dict[str, Any]:
        tid = str(config.get("configurable", {}).get("thread_id", "default"))
        msgs = state["messages"]
        if not msgs:
            return {}
        last = msgs[-1]
        if isinstance(last, AIMessage) and last.content:
            text = last.content if isinstance(last.content, str) else str(last.content)
            if text.strip():
                save_fetch(thread_id=tid, raw_summary=text, sport=None, query="chat_turn")
        return {}

    graph_builder = StateGraph(SportsNewsState)
    graph_builder.add_node("sports_chat", sports_chat)
    graph_builder.add_node("tools", tool_node)
    graph_builder.add_node("persist", persist_after_turn)
    graph_builder.add_edge(START, "sports_chat")

    def route_after_chat(state: SportsNewsState) -> Literal["tools", "persist"]:
        last = state["messages"][-1]
        if isinstance(last, AIMessage) and last.tool_calls:
            return "tools"
        return "persist"

    graph_builder.add_conditional_edges(
        "sports_chat",
        route_after_chat,
        {"tools": "tools", "persist": "persist"},
    )
    graph_builder.add_edge("tools", "sports_chat")
    graph_builder.add_edge("persist", END)
    return graph_builder.compile(checkpointer=checkpointer)

## 3. Compile and run

In [16]:
init_news_db()
checkpointer = make_checkpointer()
graph = build_graph(checkpointer)

THREAD_ID = "sports-demo-1"
config = {"configurable": {"thread_id": THREAD_ID}}

result = graph.invoke(
    {"messages": [HumanMessage(content="What are today's headlines in the Premier League?")]},
    config=config,
)
result["messages"][-1].content

"Here are today's headlines in the Premier League:\n\n1. **Casemiro: Man Utd Decision 'Made and Done'** - Casemiro has confirmed he will not reverse his decision to leave Manchester United at the end of the season.\n2. **Transfer Battle Over Elite Midfielders** - Premier League clubs are preparing for a competitive transfer window focused on top midfield talent.\n3. **Everton Update** - David Moyes is expected to be offered a new contract at the end of the season.\n4. **Shay Given's Concerns About Liverpool** - Following their latest Premier League loss, Given has expressed worries regarding Liverpool's performance.\n5. **Aston Villa Solidify Fourth Spot** - Aston Villa has strengthened their position in the top four, while West Ham remains in the bottom three.\n\nThese headlines highlight key developments and discussions within the Premier League."

## 4. Inspect stored rows

In [17]:
list_for_thread(THREAD_ID, limit=5)

[{'id': 3,
  'thread_id': 'sports-demo-1',
  'sport': None,
  'query': 'chat_turn',
  'raw_summary': "Here are today's headlines in the Premier League:\n\n1. **Casemiro: Man Utd Decision 'Made and Done'** - Casemiro has confirmed he will not reverse his decision to leave Manchester United at the end of the season.\n2. **Transfer Battle Over Elite Midfielders** - Premier League clubs are preparing for a competitive transfer window focused on top midfield talent.\n3. **Everton Update** - David Moyes is expected to be offered a new contract at the end of the season.\n4. **Shay Given's Concerns About Liverpool** - Following their latest Premier League loss, Given has expressed worries regarding Liverpool's performance.\n5. **Aston Villa Solidify Fourth Spot** - Aston Villa has strengthened their position in the top four, while West Ham remains in the bottom three.\n\nThese headlines highlight key developments and discussions within the Premier League.",
  'created_at': '2026-03-28T05:02:21

In [18]:
list_recent(5)

[{'id': 3,
  'thread_id': 'sports-demo-1',
  'sport': None,
  'query': 'chat_turn',
  'raw_summary': "Here are today's headlines in the Premier League:\n\n1. **Casemiro: Man Utd Decision 'Made and Done'** - Casemiro has confirmed he will not reverse his decision to leave Manchester United at the end of the season.\n2. **Transfer Battle Over Elite Midfielders** - Premier League clubs are preparing for a competitive transfer window focused on top midfield talent.\n3. **Everton Update** - David Moyes is expected to be offered a new contract at the end of the season.\n4. **Shay Given's Concerns About Liverpool** - Following their latest Premier League loss, Given has expressed worries regarding Liverpool's performance.\n5. **Aston Villa Solidify Fourth Spot** - Aston Villa has strengthened their position in the top four, while West Ham remains in the bottom three.\n\nThese headlines highlight key developments and discussions within the Premier League.",
  'created_at': '2026-03-28T05:02:21